# Tool 2 — Call Record Lookup: manual test notebook

Exercises the two-stage structured-filter pipeline in `src/tools/call_lookup.py`.

**Note:** cells (a)–(e) need **no** LLM — they cover the deterministic Stage-1
regex/enum parsing, the `_needs_llm` decision, deterministic `lookup_calls`
output, and the Stage-2 merge/fallback logic (with a *mocked* LLM). Only cell (f)
makes a real Stage-2 call and needs a working provider in `config.json`
(a reachable `local_lmstudio` server, or a valid key for a cloud provider).


### (a) Imports + config — which provider/model would Stage 2 use

In [1]:
import os, sys

# Make `import src...` work when the kernel's cwd is notebooks/.
sys.path.insert(0, os.path.abspath(".."))

from src.config import CONFIG
import src.tools.call_lookup as cl

prov = CONFIG["active_provider"]
pconf = CONFIG["providers"][prov]
print(f"Active provider : {prov}")
print(f"Model           : {pconf['model']}")
print(f"Records loaded  : {len(cl._load_df())} rows from data/calls.csv")
print(f"Columns         : {list(cl._load_df().columns)}")


Active provider : local_lmstudio
Model           : qwen/qwen3.8-27b
Records loaded  : 50 rows from data/calls.csv
Columns         : ['call_id', 'date', 'agent_name', 'call_type', 'direction', 'duration_mins', 'transcript', 'channel']


### (b) Stage 1 — `_regex_filters()` on a range of queries (no LLM)
Pure regex/enum prefilter. It returns **only** the fields it could resolve
deterministically; an empty dict means nothing matched.

In [2]:
stage1_queries = [
    "show me call 1042",                 # call_id
    "#1006",                             # call_id (hash form)
    "last 3 escalation calls by Priya",  # agent + call_type + limit(desc)
    "first 2 calls",                     # limit(asc)
    "inbound chat calls",                # direction + channel
    "today's calls",                     # simple date
    "calls from the last 40 days",       # simple relative range
    "asdf qwerty",                       # nothing -> {}
]
for q in stage1_queries:
    print(f"{q!r:36} -> {cl._regex_filters(q)}")


'show me call 1042'                  -> {'call_id': 'CALL-1042'}
'#1006'                              -> {'call_id': 'CALL-1006'}
'last 3 escalation calls by Priya'   -> {'agent_name': 'Priya', 'call_type': 'escalation', 'limit': 3, 'sort_order': 'desc'}
'first 2 calls'                      -> {'limit': 2, 'sort_order': 'asc'}
'inbound chat calls'                 -> {'direction': 'inbound', 'channel': 'chat'}
"today's calls"                      -> {'date_from': '2026-08-25', 'date_to': '2026-08-25'}
'calls from the last 40 days'        -> {'date_from': '2026-07-16', 'date_to': '2026-08-25'}
'asdf qwerty'                        -> {}


### (c) `_needs_llm()` — which queries escalate to Stage 2 (no LLM)
Escalates **only** on fuzzy date language (`this week`, `last month`, `recent`)
or when Stage 1 matched nothing at all.

In [3]:
needs_llm_queries = [
    "last 3 escalation calls by Priya",  # deterministic       -> False
    "inbound chat calls",                # deterministic       -> False
    "calls from the last 40 days",       # regex resolves date -> False
    "recent repair calls",               # fuzzy "recent"      -> True
    "calls from this month",             # fuzzy + empty regex  -> True
    "asdf qwerty",                       # empty regex         -> True
]
for q in needs_llm_queries:
    rf = cl._regex_filters(q)
    print(f"needs_llm={cl._needs_llm(q, rf)!s:5}  {q!r}")


needs_llm=False  'last 3 escalation calls by Priya'
needs_llm=False  'inbound chat calls'
needs_llm=False  'calls from the last 40 days'
needs_llm=True   'recent repair calls'
needs_llm=True   'calls from this month'
needs_llm=True   'asdf qwerty'


### (d) `lookup_calls()` — deterministic queries end-to-end (no LLM)
These never reach Stage 2, so they run with the LLM provider down.

In [4]:
for q in ["show me call 1042",
          "last 3 escalation calls",
          "inbound chat calls by Amit",
          "outbound email calls by James"]:   # -> no matches
    print("=" * 70)
    print("QUERY:", q)
    print("-" * 70)
    print(cl.lookup_calls(q))
    print()


QUERY: show me call 1042
----------------------------------------------------------------------
[call_lookup] stage 1 -> {'call_id': 'CALL-1042'}
[call_lookup] stage 2 skipped (deterministic answer)
[call_lookup] filters -> call_id=CALL-1042, sort_order=desc
[call_lookup] query='show me call 1042' matched 1 record(s)
[CALL-1042] 2026-08-12 — Sara — escalation (inbound, phone, 11 min)

QUERY: last 3 escalation calls
----------------------------------------------------------------------
[call_lookup] stage 1 -> {'call_type': 'escalation', 'limit': 3, 'sort_order': 'desc'}
[call_lookup] stage 2 skipped (deterministic answer)
[call_lookup] filters -> call_type=escalation, limit=3, sort_order=desc
[call_lookup] query='last 3 escalation calls' matched 3 record(s)
[CALL-1035] 2026-08-19 — Amit — escalation (inbound, phone, 9 min)
[CALL-1005] 2026-08-18 — James — escalation (inbound, phone, 12 min)
[CALL-1039] 2026-08-15 — Rahul — escalation (inbound, phone, 12 min)

QUERY: inbound chat calls 

### (e) Stage-2 logic with a **mocked** LLM (no real call)
Verifies the two guarantees of the merge without needing a live model:
1. a regex-confirmed field is **never** overridden by the LLM;
2. if the LLM call fails, the tool falls back to the regex-only result.

In [5]:
from src.tools.call_lookup import CallFilters

# --- 1) merge precedence: regex call_type=repair must survive an LLM "device" ---
cl._structured_llm.cache_clear()
cl._structured_llm = lambda: type("M", (), {"invoke": staticmethod(
    lambda prompt: CallFilters(call_type="device",
                               date_from="2026-08-01", date_to="2026-08-31"))})()
merged = cl._resolve_filters("recent repair calls")   # regex -> call_type=repair
print("merged filters :", merged.describe())
assert merged.call_type.value == "repair", "regex field was overridden!"
print("OK: regex call_type=repair preserved; LLM only added the date range\n")

# --- 2) LLM failure -> graceful fallback to the regex-only result ---
cl._structured_llm = lambda: (_ for _ in ()).throw(RuntimeError("simulated LLM outage"))
fallback = cl._resolve_filters("recent repair calls")
print("fallback filters:", fallback.describe())
assert fallback.call_type.value == "repair"
print("OK: LLM outage fell back to the regex result, no exception raised")


[call_lookup] stage 1 -> {'call_type': 'repair'}
[call_lookup] stage 2 -> querying structured-output LLM
merged filters : call_type=repair, date_from=2026-08-01, date_to=2026-08-31, sort_order=desc
OK: regex call_type=repair preserved; LLM only added the date range

[call_lookup] stage 1 -> {'call_type': 'repair'}
[call_lookup] stage 2 -> querying structured-output LLM
[call_lookup] LLM extraction failed, using regex result: simulated LLM outage
fallback filters: call_type=repair, sort_order=desc
OK: LLM outage fell back to the regex result, no exception raised


### (f) Live Stage-2 call (needs a working provider)
Uses the real `.with_structured_output()` model to resolve a fuzzy date.
Re-import the module first to drop the mocks from cell (e). If the local model
can't produce valid structured output, the tool logs it and falls back to the
regex result — so this cell returns records either way, it just may ignore the
fuzzy date.

In [7]:
import importlib
importlib.reload(cl)   # restore the real _structured_llm (undo cell (e) mocks)

print(cl.lookup_calls("repair calls from this month"))


[call_lookup] stage 1 -> {'call_type': 'repair'}
[call_lookup] stage 2 -> querying structured-output LLM
[call_lookup] filters -> call_type=repair, date_from=2026-08-01, date_to=2026-08-31, sort_order=desc
[call_lookup] query='repair calls from this month' matched 8 record(s)
[CALL-1032] 2026-08-22 — Sara — repair (inbound, phone, 6 min)
[CALL-1002] 2026-08-21 — Priya — repair (inbound, phone, 9 min)
[CALL-1036] 2026-08-18 — Priya — repair (inbound, email, 11 min)
[CALL-1006] 2026-08-17 — Rahul — repair (outbound, phone, 5 min)
[CALL-1043] 2026-08-11 — James — repair (inbound, phone, 10 min)
[CALL-1013] 2026-08-11 — Amit — repair (inbound, phone, 8 min)
[CALL-1049] 2026-08-05 — Rahul — repair (inbound, chat, 8 min)
[CALL-1019] 2026-08-05 — Sara — repair (inbound, phone, 10 min)


### (g) Parity guard — `resolve_filters_with_status()` vs `_resolve_filters()`

`resolve_filters_with_status()` (added for `agent/graph.py`'s `lookup_node`, so it
can report whether Stage-2 extraction degraded to keyword-only) **deliberately
duplicates** `_resolve_filters()`'s orchestration — no shared helper was
extracted, to keep `_resolve_filters`'s signature stable for existing callers.
This cell is the **guard against the two drifting apart**: for a regex-only query
*and* a fuzzy-date query, the returned `CallFilters` must be identical
(`.model_dump()`), and the `llm_degraded` flag must be correct. Stage 2 is
**mocked** so the fuzzy-date comparison tests the shared orchestration, not the
LLM's run-to-run determinism.